# Task 8 — Flow Assignment Pipeline

Orchestrates the full Task 8 pipeline using OOP modules in `task8_pipeline/`.

| Step | Class | Output |
|------|-------|--------|
| 8.1 | `RoutingTableBuilder` | `county_routing_lookup.parquet` |
| 8.2 | `AreaFlowMatrixBuilder` | `area_flow_matrix.parquet` |
| 8.3 | `HubThroughputCalculator` | `hub_throughput.csv` (initial) |
| 8.4 | `LinkFlowLoader` | `hub_link_flows.csv` |
| 8.5 | `GatewayThroughputCalculator` | `gateway_throughput.csv` |
| 8.6 | `InterfaceNodeRouter` | `interface_hub_routing.csv`, updated `hub_throughput.csv` |
| 8.7 | `FlowAnalyzer` | Analysis tables (printed) |
| 8.8 | `FigureGenerator` | 5 PNG figures |

**Interpreter:** `~/.venvs/general/bin/python3`

In [1]:
import sys
from pathlib import Path

# Ensure the Task8 directory is on sys.path so task8_pipeline is importable
TASK8_DIR = Path("__file__").resolve().parent if "__file__" in dir() else Path.cwd()
# When run from the notebook, cwd is typically the project root or Task8/
# We locate the package relative to this notebook's own location.
NB_DIR = Path(globals().get("__vsc_ipynb_file__", __file__)).resolve().parent \
    if "__vsc_ipynb_file__" in globals() \
    else Path.cwd() / "Task8"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

print(f"Notebook dir : {NB_DIR}")
print(f"sys.path[0]  : {sys.path[0]}")

Notebook dir : /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Task8/Task8
sys.path[0]  : /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Task8/Task8


In [2]:
from task8_pipeline import (
    Task8Config,
    RoutingTableBuilder,
    AreaFlowMatrixBuilder,
    HubThroughputCalculator,
    LinkFlowLoader,
    GatewayThroughputCalculator,
    InterfaceNodeRouter,
    FlowAnalyzer,
    FigureGenerator,
)

cfg = Task8Config()
print(f"Project root : {cfg.ROOT}")
print(f"Output dir   : {cfg.OUT_DIR}")
print(f"Figure dir   : {cfg.FIG_DIR}")

Project root : /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2
Output dir   : /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8
Figure dir   : /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/figures


## 8.1 — County Routing Lookup Table

In [3]:
print("=" * 60)
print("STEP 8.1 — County Routing Lookup Table")
print("=" * 60)
lookup = RoutingTableBuilder(cfg).run()
display(lookup.head(3))

STEP 8.1 — County Routing Lookup Table
  [8.1] county_area: 434 rows | areas: 132 | regions: 50
  [8.1] gw_shares: 319 rows | unique gateways: 312 ✓
  [8.1] hub_shares: 52 rows | multi-hub regions: {0: 2, 7: 2} ✓
  [8.1] lookup: 1,112 rows | county × gateway × hub combos
  [8.1] ✓ combined_share sums to 1.0 for all 434 counties
  [8.1] Saved → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/county_routing_lookup.parquet  (19.5 KB)


,fips,area_id,region_id,gateway_candidate_id,hub_candidate_id,gw_share,hub_share,combined_share
0,42003,0_0,0,T4-PA-00068,T4-PA-00319,0.213927,0.540936,0.115721
1,42003,0_0,0,T4-PA-00068,T4-PA-00350,0.213927,0.459064,0.098206
2,42003,0_0,0,T4-PA-00090,T4-PA-00319,0.167585,0.540936,0.090653


## 8.3 — Hub-Level Throughput

> **Note:** 8.3 runs before 8.2 because the region_flow_matrix total is needed
> as the validation target for the area-flow matrix (Task 8.2).

In [4]:
print("=" * 60)
print("STEP 8.3 — Hub-Level Throughput")
print("=" * 60)
hub_tp, rfm_total_25, region_hubs = HubThroughputCalculator(cfg).run()
display(hub_tp.head(5))

STEP 8.3 — Hub-Level Throughput
  [8.3] region_flow_matrix: 2,500 rows | total_2025 = 2,454,583.5 ktons ✓
  [8.3] hub_assign: 52 rows | multi-hub regions: {0: [('T4-PA-00319', '0.5409'), ('T4-PA-00350', '0.4591')], 7: [('T4-NY-00171', '0.4256'), ('T4-NY-00249', '0.5744')]}
  [8.3] Accumulation complete. 50 hub rows built.
  [8.3] ✓ sum(throughput)×0.5 = 2,454,583.5 ktons ≈ RFM total | inbound≈outbound ✓
  [8.3] Saved → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/hub_throughput.csv  (6.9 KB)


,candidate_id,facility_name,source_state,region_id,inbound_ktons_2025,outbound_ktons_2025,internal_ktons_2025,throughput_ktons_2025,throughput_ktons_2030
13,T4-NJ-00197,"1075 Secaucus Rd, Jersey City, NJ",NJ,34,204777.044095,204777.044095,91979.833305,409554.088190,494320.890148
7,T4-ME-00014,"35 Canal St, Lewiston, ME",ME,15,152621.355551,152621.355551,115507.013504,305242.711102,336996.276307
42,T4-VA-00091,Plaza 500 - Building A,VA,2,84396.506831,84396.506831,31696.925096,168793.013662,193981.087339
37,T4-RI-00028,Collyer Business Center,RI,44,84138.790219,84138.790219,36313.728391,168277.580437,200528.551622
3,T4-MA-00113,"613 Main St, Wilmington, MA",MA,28,80571.883480,80571.883480,34061.480002,161143.766959,190877.682574


## 8.2 — Area-Pair Flow Matrix

Loads `raw.parquet` (~33M rows). May take 60–90 seconds on first run.

In [5]:
print("=" * 60)
print("STEP 8.2 — Area-Pair Flow Matrix")
print("=" * 60)
area_flow = AreaFlowMatrixBuilder(cfg, rfm_total_25).run()
display(area_flow.head(3))

STEP 8.2 — Area-Pair Flow Matrix
  [8.2] fips_to_area: 434 NE counties | areas: 132
  [8.2] Loading raw.parquet …


  [8.2] Loaded 33,404,629 rows → 27,284,712 after commodity filter


  [8.2] internal: 2,215,787 | inbound: 12,523,330 | outbound: 12,545,595


  [8.2] area_flow_matrix: 17,424 rows (max possible: 132×132 = 17,424)
  [8.2] directed internal total =     1,227,291.7 ktons
         expected (rfm/2)        =     1,227,291.7 ktons
         deviation               =            -0.0 ktons (0.0000%)
  [8.2] ✓ Validation passed | 2030/2025 ratio = 1.1286 (growth confirmed)
  [8.2] Saved → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/area_flow_matrix.parquet  (341.4 KB)


,origin_area_id,dest_area_id,tons_2025,tons_2030
0,8_0,8_0,4102.632178,4514.275778
1,8_0,48_3,592.619359,649.624919
2,8_0,8_3,787.164181,865.034168


## 8.4 — Hub-to-Hub Link Flow Loading

In [6]:
print("=" * 60)
print("STEP 8.4 — Hub-to-Hub Link Flow Loading")
print("=" * 60)
hub_links = LinkFlowLoader(cfg, rfm_total_25, region_hubs).run()
display(hub_links.head(5))

STEP 8.4 — Hub-to-Hub Link Flow Loading
  [8.4] Hub network: 133 links
  [8.4] Assignments: 2,648 total | direct=288 (10.9%) | nearest-neighbor=2,360 (89.1%)
  [8.4] ✓ link total=1,829,086.8 ktons ≈ inter-region RFM total | zero-flow links: 1
  [8.4] Saved → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/hub_link_flows.csv  (18.8 KB)


,hub_a_candidate_id,hub_b_candidate_id,hub_a_name,hub_b_name,distance_miles,flow_ktons_2025,flow_ktons_2030,flow_intensity_original_ktons
6,T4-NJ-00197,T4-NJ-00221,"1075 Secaucus Rd, Jersey City, NJ",Fairlawn Industries,12.992,66721.184553,81638.029203,24403.500723
62,T4-CT-00017,T4-NJ-00197,Shepard's Warehouse,"1075 Secaucus Rd, Jersey City, NJ",53.901,56089.844363,69362.089372,51468.505941
46,T4-MA-00113,T4-RI-00028,"613 Main St, Wilmington, MA",Collyer Business Center,47.612,47145.868044,55770.492810,47145.868044
24,T4-NJ-00018,T4-NJ-00197,Spring Brook Commerce Center,"1075 Secaucus Rd, Jersey City, NJ",29.207,37272.658233,43102.435795,10761.164332
13,T4-MD-00001,T4-MD-00077,Snowden Distribution Center,"4311 Erdman Ave, Baltimore, MD",17.374,31209.101295,35718.208018,22435.870629


## 8.5 — Gateway-Level Throughput

In [7]:
print("=" * 60)
print("STEP 8.5 — Gateway-Level Throughput")
print("=" * 60)
gw_tp = GatewayThroughputCalculator(cfg).run()
display(gw_tp.head(5))

STEP 8.5 — Gateway-Level Throughput
  [8.5] area_flow_matrix: 17,424 rows | total_2025 = 1,227,291.7 ktons (directed)
  [8.5] gaa: 319 rows | gateways: 312 | areas: 132 ✓
  [8.5] Accumulation complete. 312 gateway rows built.
  [8.5] ✓ sum(gw_throughput)×0.5 = 1,227,291.7 ktons ≈ directed area_flow total ✓
  [8.5] Saved → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/gateway_throughput.csv  (38.0 KB)


,candidate_id,facility_name,source_state,area_id,region_id,inbound_ktons_2025,outbound_ktons_2025,throughput_ktons_2025,throughput_ktons_2030
55,T4-ME-00053,"70 Bennett St, Bangor, ME",ME,15_1,15,15386.837532,16244.686075,31631.523607,34829.862058
54,T4-ME-00052,"175 Allied Rd, Auburn, ME",ME,15_2,15,12132.406820,16083.466851,28215.873672,31337.666355
56,T4-ME-00054,"2879 Hotel Rd, Auburn, ME",ME,15_2,15,11258.946866,14925.554458,26184.501323,29081.543804
50,T4-MD-00183,"8711 Westphalia Rd, Upper Marlboro, MD",MD,2_2,2,10389.246020,8933.275718,19322.521739,21912.015842
57,T4-NH-00023,"7 Symmes Dr, Londonderry, NH",NH,15_3,15,10186.093626,8868.221949,19054.315575,21089.097597


## 8.6 — Interface Node Flow Routing

In [8]:
print("=" * 60)
print("STEP 8.6 — Interface Node Flow Routing")
print("=" * 60)
iface_routing, hub_tp_updated = InterfaceNodeRouter(cfg, hub_tp).run()
display(iface_routing)

STEP 8.6 — Interface Node Flow Routing
  [8.6] Interface nodes: 29 rows | classes: {'national': 12, 'global': 9, 'continental': 8}
  [8.6] ✓ Continental range: [2738.8, 30162.1] ktons
  [8.6] ✓ All 29 interface nodes assigned | hub interface sum = 794,338.1 ktons ✓
  [8.6] Saved → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/interface_hub_routing.csv  (3.7 KB)
  [8.6] Updated → /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/hub_throughput.csv  (added interface_throughput_ktons_2025/2030 columns)


,node_name,facility_name,interface_class,nearest_hub_candidate_id,nearest_hub_name,distance_miles,tons_2025_ktons,tons_2030_ktons
0,IF_hampton_roads_va,"Hampton Roads, VA",global,T4-VA-00011,"2034 Atlantic Ave, Chesapeake, VA",4.025348,20204.535156,20404.755859
1,IF_philadelphia_pa,"Philadelphia, PA",global,T4-PA-00074,"300-332 E Allegheny Ave, Philadelphia, PA",5.934589,20204.535156,20404.755859
2,IF_washington_dulles_intl,WASHINGTON DULLES INTL,global,T4-VA-00091,Plaza 500 - Building A,19.594876,26581.482281,30678.560272
3,IF_baltimore_washington_intl_thurgood_marshall,BALTIMORE/WASHINGTON INTL THURGOOD MARSHALL,global,T4-MD-00001,Snowden Distribution Center,8.893161,15530.753693,17924.552069
4,IF_general_edward_lawrence_logan_intl,GENERAL EDWARD LAWRENCE LOGAN INTL,global,T4-MA-00113,"613 Main St, Wilmington, MA",14.533334,19353.708448,22336.749501
5,IF_newark_liberty_intl,NEWARK LIBERTY INTL,global,T4-NJ-00096,"1200 Fuller Rd, Linden, NJ",6.764828,73352.944363,84659.038234
6,IF_john_f_kennedy_intl,JOHN F KENNEDY INTL,global,T4-NJ-00197,"1075 Secaucus Rd, Jersey City, NJ",16.733887,236425.858134,272866.834959
7,IF_philadelphia_intl,PHILADELPHIA INTL,global,T4-PA-00017,"1260 E Woodland Ave, Springfield Township, PA",5.171743,92049.582462,106237.441302
8,IF_pittsburgh_intl,PITTSBURGH INTL,global,T4-PA-00350,Building 7 & 8,13.019485,15948.889369,18407.136163
9,IF_buffalo_niagara_falls,Buffalo Niagara Falls,continental,T4-NY-00171,FedEx,11.611962,30162.118000,30162.118000


## 8.7 — Analysis: Critical Hubs, Corridors & Concentration

In [9]:
print("=" * 60)
print("STEP 8.7 — Flow Analysis")
print("=" * 60)
hub_tp_a, gw_tp_a, hub_links_a, iface_a = FlowAnalyzer(cfg).run()

STEP 8.7 — Flow Analysis
  [8.7] Loaded: 50 hubs | 312 gateways | 133 links | 29 interface nodes

Top 10 Regional Hubs by Throughput (2025)
   1. 1075 Secaucus Rd, Jersey City, NJ             (NJ) |    409,554 ktons | iface=36.6% [multi-region]
   2. 35 Canal St, Lewiston, ME                     (ME) |    305,243 ktons | iface=4.7%
   3. Plaza 500 - Building A                        (VA) |    168,793 ktons | iface=13.6%
   4. Collyer Business Center                       (RI) |    168,278 ktons | iface=0.0%
   5. 613 Main St, Wilmington, MA                   (MA) |    161,144 ktons | iface=10.7%
   6. Shepard's Warehouse                           (CT) |    150,257 ktons | iface=0.0%
   7. 2600 7th Ave, Watervliet, NY                  (NY) |    149,231 ktons | iface=6.1%
   8. 61 Chapel Rd, Manchester, CT                  (CT) |    146,932 ktons | iface=0.0%
   9. Snowden Distribution Center                   (MD) |    134,143 ktons | iface=10.4%
  10. 5107 North Point Blvd, Sparrows Po

## 8.8 — Figures & Exports

In [10]:
print("=" * 60)
print("STEP 8.8 — Figure Generation")
print("=" * 60)
FigureGenerator(cfg, hub_tp_a, gw_tp_a, hub_links_a).run()

STEP 8.8 — Figure Generation


  [8.8] hub_geo: 50 | gw_geo: 312 | NE counties: 434



  [8.8] 5 figures saved to /Users/tianyihu/Documents/Dev/Python/Projects/ISYE6339_Case2/Data/Task8/figures
    ✓ fig_gateway_throughput_map.png  (343 KB)
    ✓ fig_hub_link_flow_map.png  (385 KB)
    ✓ fig_hub_throughput_bar.png  (135 KB)
    ✓ fig_hub_throughput_map.png  (331 KB)
    ✓ fig_top_corridors_bar.png  (129 KB)


## Final Validation

In [11]:
import os
import pandas as pd

exports = {
    "hub_throughput.csv":            (cfg.HUB_THROUGHPUT,         50),
    "gateway_throughput.csv":        (cfg.GATEWAY_THROUGHPUT,     312),
    "hub_link_flows.csv":            (cfg.HUB_LINK_FLOWS,         133),
    "interface_hub_routing.csv":     (cfg.INTERFACE_HUB_ROUTING,  29),
    "area_flow_matrix.parquet":      (cfg.AREA_FLOW_MATRIX,       None),
    "county_routing_lookup.parquet": (cfg.COUNTY_ROUTING_LOOKUP,  None),
}

print("Tabular Export Verification")
print("=" * 60)
all_ok = True
for name, (path, expected_rows) in exports.items():
    exists   = path.exists()
    size_kb  = os.path.getsize(path) / 1024 if exists else 0
    if exists and expected_rows is not None:
        df  = pd.read_csv(path) if name.endswith(".csv") else pd.read_parquet(path)
        n   = len(df)
        ok  = n == expected_rows
        all_ok = all_ok and ok
        flag = "✓" if ok else "✗"
        print(f"  {flag} {name:<42} {n:>4} rows  ({size_kb:.0f} KB)")
    elif exists:
        print(f"  ✓ {name:<42} {size_kb:.0f} KB")
    else:
        print(f"  ✗ {name:<42} MISSING")
        all_ok = False

figs = sorted(cfg.FIG_DIR.glob("fig_*.png"))
print(f"\nFigures ({len(figs)}/5):")
for f in figs:
    print(f"  ✓ {f.name}  ({os.path.getsize(f)//1024} KB)")

assert all_ok,       "One or more tabular exports have unexpected row counts"
assert len(figs) == 5, f"Expected 5 figures, found {len(figs)}"
print("\n✓ All exports verified — Task 8 complete.")

Tabular Export Verification
  ✓ hub_throughput.csv                           50 rows  (8 KB)
  ✓ gateway_throughput.csv                      312 rows  (38 KB)
  ✓ hub_link_flows.csv                          133 rows  (19 KB)
  ✓ interface_hub_routing.csv                    29 rows  (4 KB)
  ✓ area_flow_matrix.parquet                   341 KB
  ✓ county_routing_lookup.parquet              20 KB

Figures (5/5):
  ✓ fig_gateway_throughput_map.png  (343 KB)
  ✓ fig_hub_link_flow_map.png  (385 KB)
  ✓ fig_hub_throughput_bar.png  (135 KB)
  ✓ fig_hub_throughput_map.png  (331 KB)
  ✓ fig_top_corridors_bar.png  (129 KB)

✓ All exports verified — Task 8 complete.
